# Visualization and Reporting

Reads the completed experiment outputs, regenerates paper-facing assets, and displays the Verdict Matrix in the final cell.

In [1]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

current = Path.cwd().resolve()
repo_root = next((p for p in [current, *current.parents] if (p / "evaluation").exists() and (p / "visualization").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from {current}")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluation.paper_assets import PaperPaths, generate_paper_assets
from evaluation.verdict_matrix import build_verdict_display_table, plot_verdict_matrix
from visualization.ac_gate_architecture import draw_ac_gate_architecture
from visualization.compact_paper_figures import build_figures as build_compact_figures
from visualization.panel_r2_floor_figure import build_figure as build_r2_floor_figure
from visualization.paper_figures import plot_workflow_overview
from visualization.proxy_shuffle_control_figure import build_figure as build_proxy_shuffle_figure

PLAN_NAME = "complete_20seed_20260426"
OUTPUT_ROOT = repo_root / "outputs" / "paper_assets" / PLAN_NAME
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
COMPACT_DIR = OUTPUT_ROOT / "compact_figures"
PROXY_ROOT = repo_root / "outputs" / "negative_controls" / "proxy_shuffle_20seed_20260511"
FORMATS = ["png", "svg"]

display(pd.Series({"plan_name": PLAN_NAME, "paper_assets_root": OUTPUT_ROOT, "proxy_root": PROXY_ROOT}).to_frame("value"))

,value
plan_name,complete_20seed_20260426
paper_assets_root,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
proxy_root,C:\DevSpace\PyDevspace\CMDL\outputs\negative_c...


In [2]:
bundle = generate_paper_assets(
    paths=PaperPaths(workspace_root=repo_root, plan_name=PLAN_NAME),
    output_root=OUTPUT_ROOT,
    include_figures=True,
    n_boot=5000,
    bootstrap_seed=0,
    proxy_root=PROXY_ROOT,
)
table_rows = pd.DataFrame([
    {"table": name, "rows": len(frame), "path": TABLES_DIR / name}
    for name, frame in bundle.tables.items()
])
figure_rows = pd.DataFrame([
    {"figure": name, "path": path}
    for name, path in bundle.figure_paths.items()
])
required_tables = [
    "synthetic_significance_kstar_mae.csv",
    "synthetic_significance_task_loss.csv",
    "economics_significance_test_r2.csv",
    "energy_significance_test_r2.csv",
    "real_r2_wilcoxon.csv",
    "proxy_shuffle_compact_table.csv",
    "verdict_matrix.csv",
]
required_table_rows = pd.DataFrame([
    {
        "table": name,
        "exists": (TABLES_DIR / name).exists(),
        "rows": len(bundle.tables.get(name, [])),
        "path": TABLES_DIR / name,
    }
    for name in required_tables
])
missing_required = required_table_rows.loc[(~required_table_rows["exists"]) | (required_table_rows["rows"] <= 0)]
if not missing_required.empty:
    raise AssertionError(f"Missing or empty required paper tables: {missing_required['table'].tolist()}")

display(required_table_rows)
display(table_rows)
display(figure_rows)
print(f"Manifest: {bundle.manifest_path}")

,table,exists,rows,path
0,synthetic_significance_kstar_mae.csv,True,12,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
1,synthetic_significance_task_loss.csv,True,12,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
2,economics_significance_test_r2.csv,True,7,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
3,energy_significance_test_r2.csv,True,7,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
4,real_r2_wilcoxon.csv,True,14,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
5,proxy_shuffle_compact_table.csv,True,4,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
6,verdict_matrix.csv,True,12,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...


,table,rows,path
0,synthetic_main_table.csv,14,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
1,synthetic_significance_kstar_mae.csv,12,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
2,synthetic_significance_task_loss.csv,12,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
3,synthetic_bootstrap_ci.csv,42,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
4,realdata_forecast_table.csv,16,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
5,economics_significance_test_r2.csv,7,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
6,energy_significance_test_r2.csv,7,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
7,real_r2_wilcoxon.csv,14,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
8,baseline_compact_table.csv,30,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
9,economics_stratified_main_table.csv,21,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...


,figure,path
0,workflow_overview.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
1,synthetic_kstar_mae_seed_distribution_linear.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
2,synthetic_kstar_mae_seed_distribution_nonlinea...,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
3,economics_forecast_seed_distribution.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
4,energy_forecast_seed_distribution.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
5,economics_structured_mechanism_seed_distributi...,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
6,energy_structured_mechanism_seed_distribution.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
7,economics_case_study.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
8,energy_case_study.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
9,realdata_case_study_panel.png,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...


Manifest: C:\DevSpace\PyDevspace\CMDL\outputs\paper_assets\complete_20seed_20260426\paper_artifacts_manifest.json


In [3]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
COMPACT_DIR.mkdir(parents=True, exist_ok=True)
created = []

workflow_fig = plot_workflow_overview(FIGURES_DIR / "workflow_overview.svg")
plt.close(workflow_fig)
created.append(FIGURES_DIR / "workflow_overview.svg")

architecture_fig = draw_ac_gate_architecture()
for fmt in FORMATS:
    target = repo_root / "outputs" / "paper_assets" / f"ac_gate_architecture.{fmt}"
    target.parent.mkdir(parents=True, exist_ok=True)
    architecture_fig.savefig(target, dpi=300 if fmt == "png" else None, bbox_inches="tight")
    created.append(target)
plt.close(architecture_fig)

created.extend(build_compact_figures(root=repo_root, plan=PLAN_NAME, output_dir=COMPACT_DIR, formats=FORMATS, n_boot=5000, tft_plan=PLAN_NAME))
created.extend(build_r2_floor_figure(output_dir=COMPACT_DIR, formats=FORMATS, plan=PLAN_NAME))
if (PROXY_ROOT / "comparison" / "proxy_shuffle_summary.csv").exists():
    created.extend(build_proxy_shuffle_figure(proxy_root=PROXY_ROOT, output_dir=COMPACT_DIR, formats=FORMATS))
else:
    print(f"Skipping proxy-shuffle figure; missing {PROXY_ROOT / 'comparison' / 'proxy_shuffle_summary.csv'}")

display(pd.DataFrame({"created": created}))

,created
0,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
1,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
2,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
3,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
4,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
5,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
6,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
7,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
8,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...
9,C:\DevSpace\PyDevspace\CMDL\outputs\paper_asse...


## Verdict Matrix

In [4]:
verdict = pd.read_csv(TABLES_DIR / "verdict_matrix.csv", keep_default_na=False)
display(build_verdict_display_table(verdict))

verdict_png = FIGURES_DIR / "verdict_matrix.png"
verdict_svg = FIGURES_DIR / "verdict_matrix.svg"
fig = plot_verdict_matrix(verdict, save_path=verdict_png)
fig.savefig(verdict_svg, bbox_inches="tight")
plt.close(fig)
display(verdict[["domain", "layer", "verdict", "label", "evidence"]])
print(f"Saved verdict matrix: {verdict_png}")
print(f"Saved verdict matrix: {verdict_svg}")

,Domain,L0,L1,L2,L3
0,Synthetic,n/c,yes,yes,yes
1,Economics,n/c,yes,yes,n/a
2,Energy,n/c,yes,yes,n/a


,domain,layer,verdict,label,evidence
0,synthetic,L0,not_certified,n/c,Paired task-loss tests versus Plain LSTM/TFT/G...
1,synthetic,L1,certified,yes,CMDL retains nonzero seed-level lag-rank signa...
2,synthetic,L2,certified,yes,CMDL k* recovery beats Plain LSTM/TFT/GA-Net a...
3,synthetic,L3,certified,yes,Known-lag alignment requires Spearman rho>=0.8...
4,economics,L0,not_certified,n/c,CMDL mean test R2=0.0541; best competing mean ...
5,economics,L1,certified,yes,"CMDL k* std must exceed epsilon=1e-06, while N..."
6,economics,L2,certified,yes,"CMDL stratifier rows require valid seeds, Fish..."
7,economics,L3,not_claimed,n/a,No optional ground-truth lag is available for ...
8,energy,L0,not_certified,n/c,CMDL mean test R2=-0.0286; best competing mean...
9,energy,L1,certified,yes,"CMDL k* std must exceed epsilon=1e-06, while N..."


Saved verdict matrix: C:\DevSpace\PyDevspace\CMDL\outputs\paper_assets\complete_20seed_20260426\figures\verdict_matrix.png
Saved verdict matrix: C:\DevSpace\PyDevspace\CMDL\outputs\paper_assets\complete_20seed_20260426\figures\verdict_matrix.svg
